# Hướng dẫn Tinh chỉnh (Fine-tuning) PaddleOCR Recognition cho Mã Container trên Google Colab

Notebook này hỗ trợ bạn kết nối Google Drive, chuẩn bị dữ liệu mã container đã tải lên, cấu hình và huấn luyện tinh chỉnh mô hình nhận dạng ký tự **PP-OCRv3** của PaddleOCR chuyên biệt cho phông chữ Container, sau đó xuất ra mô hình suy luận tĩnh (`inference model`) lưu về Google Drive của bạn.

## Bước 1: Kết nối Google Drive và GPU

In [ ]:
# 1. Kết nối tới Google Drive của bạn
from google.colab import drive
drive.mount('/content/drive')

# 2. Kiểm tra thông tin GPU
!nvidia-smi

## Bước 2: Cài đặt thư viện PaddlePaddle GPU và PaddleOCR

In [ ]:
# 1. Cài đặt thư viện PaddlePaddle hỗ trợ GPU (phù hợp với CUDA trên Colab)
!pip install paddlepaddle-gpu

# 2. Clone mã nguồn PaddleOCR
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

# 3. Cài đặt các thư viện phụ thuộc
!pip install -r requirements.txt
!pip install -r ppocr/postprocess/make/requirements.txt

## Bước 3: Giải nén Dataset mã Container từ Google Drive

**Lưu ý cấu trúc thư mục file ZIP bạn nén tải lên Drive:**
Tạo một file `.zip` chứa cấu trúc thư mục bên dưới và tải lên Google Drive của bạn (ví dụ đặt tên là `container_rec_dataset.zip` đặt ngay tại thư mục gốc Drive):

```
container_rec_dataset.zip/
├── train/                 # Thư mục chứa các ảnh cắt từ nhãn mã container để huấn luyện
├── val/                   # Thư mục chứa các ảnh cắt để đánh giá (validation)
├── rec_train_label.txt    # File nhãn train (định dạng: tên_ảnh.jpg \t nhãn_chữ)
└── rec_val_label.txt      # File nhãn val
```

In [ ]:
# Giải nén file ZIP từ Drive trực tiếp vào thư mục train_data của PaddleOCR trên Colab
!mkdir -p train_data/rec
!unzip -q /content/drive/MyDrive/container_rec_dataset.zip -d train_data/rec/

# Xem thử các tệp giải nén thành công
!ls -la train_data/rec/

## Bước 4: Tải Trọng số Pre-trained Model PP-OCRv3 English

In [ ]:
# Tải mô hình pre-trained PP-OCRv3 tiếng Anh
!mkdir -p pretrain_models
!wget -P pretrain_models/ https://paddleocr.bj.bcebos.com/PP-OCRv3/english/en_PP-OCRv3_rec_train.tar

# Giải nén gói trọng số
%cd pretrain_models
!tar -xf en_PP-OCRv3_rec_train.tar
%cd ..

## Bước 5: Cấu hình File YAML để bắt đầu Training

Đoạn mã Python dưới đây tự động cập nhật các trường đường dẫn dữ liệu huấn luyện, đường dẫn trọng số pre-trained, số epoch chạy và cấu hình từ điển vào tệp config `en_PP-OCRv3_rec.yml` của PaddleOCR.

In [ ]:
import yaml

config_path = 'configs/rec/PP-OCRv3/en_PP-OCRv3_rec.yml'

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

# 1. Ghi đè các thông số cấu hình cốt lõi
config['Global']['pretrained_model'] = './pretrain_models/en_PP-OCRv3_rec_train/best_accuracy'
config['Global']['save_model_dir'] = './output/v3_rec_container/'
config['Global']['epoch_num'] = 150              # Huấn luyện 150 Epochs
config['Global']['print_batch_step'] = 10
config['Global']['use_gpu'] = True

# 2. Cấu hình đường dẫn dữ liệu Training
config['Train']['dataset']['data_dir'] = './train_data/rec/train/'
config['Train']['dataset']['label_file_list'] = ['./train_data/rec/rec_train_label.txt']

# 3. Cấu hình đường dẫn dữ liệu Evaluation
config['Eval']['dataset']['data_dir'] = './train_data/rec/val/'
config['Eval']['dataset']['label_file_list'] = ['./train_data/rec/rec_val_label.txt']

# Lưu lại cấu hình mới vào đè lên file
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, default_flow_style=False)

print("Đã ghi đè cấu hình file YAML huấn luyện thành công!")

## Bước 6: Khởi chạy Huấn luyện Tinh chỉnh (Fine-tuning)

In [ ]:
# Chạy lệnh python huấn luyện
!python tools/train.py -c configs/rec/PP-OCRv3/en_PP-OCRv3_rec.yml

## Bước 7: Xuất Mô hình Suy luận tĩnh (Inference Model) về Google Drive

Khi quá trình training kết thúc, tệp trọng số tốt nhất được lưu tại `./output/v3_rec_container/best_accuracy`. Ta sẽ xuất nó sang định dạng suy luận tĩnh rồi lưu trực tiếp vào Google Drive để bạn dễ dàng tải về tích hợp vào code chạy ứng dụng Web UI offline!

In [ ]:
# 1. Định nghĩa thư mục lưu trữ trên Google Drive của bạn
drive_output_dir = '/content/drive/MyDrive/paddle_rec_inference/'

# 2. Chạy script export model của PaddleOCR
!python tools/export_model.py \
  -c configs/rec/PP-OCRv3/en_PP-OCRv3_rec.yml \
  -o Global.pretrained_model=./output/v3_rec_container/best_accuracy \
  Global.save_inference_dir={drive_output_dir}

print(f"Xuất mô hình thành công! Bạn có thể lấy tệp tại thư mục: {drive_output_dir} trên Google Drive.")